# Knee MRI Abnormality Detection

## Project Overview

This project aims to develop a deep learning framework for automatically detecting abnormalities in knee MRI scans using the **RSNA Knee MRI dataset**.

The project combines medical image processing, radiology report analysis, and deep learning to identify knee abnormalities. MRI scans are analyzed across three anatomical planes: **axial, coronal, and sagittal**.

### Main Objectives

- **Data Preprocessing:** Explore, normalize, and prepare MRI volumes for model training.
- **Report Analysis:** Extract abnormality labels from radiology reports using Large Language Models (LLMs).
- **Deep Learning:** Develop CNN-based models to learn features from 3D MRI volumes.
- **Feature Fusion:** Combine information from different MRI planes to improve predictions.
- **Evaluation:** Assess model performance using appropriate classification metrics.

**Goal:** Build an automated MRI analysis pipeline capable of identifying knee abnormalities from multi-plane MRI scans.

### Import Libraries

In [1]:
import os
import sys
from pathlib import Path
import json
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## Data processing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## Model libraries pyTorch and Sickit-Learn

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import Dataset , DataLoader


from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

## medical Libraries
from dotenv import load_dotenv

from dataclasses import dataclass, asdict
import kaggle_benchmarks as kbench

from src import llm_extractor
from src import tools 


### Extract data 
the following cells build Dataframe of .csv files provided by the project owner.

In [2]:
TRAIN_SERIES_PATH =  Path("../data/train_series.csv")
TRAIN_PATH = Path("../data/train.csv")
TRAIN_SEREIS_DICOM_PATH = Path("../data/train_series")

In [3]:
study_series_df = pd.read_csv(TRAIN_SERIES_PATH)
annotation_df = pd.read_csv(TRAIN_PATH)

## Viewing & Analyzing Datasets

The `study_series` shows several Studies of MRI and each Study has several series and each series is from certain Anatomical plane.

In [ ]:
display(study_series_df.head(3))

We have downloaded 77 Studies where each study is one MRI session , and we have 440 series where each sesssion has several series where each serie is MRI images taken from a certain plane.

In [ ]:
study_series_df.StudyInstanceUID.nunique() , study_series_df.SeriesInstanceUID.nunique()

We have three types of MRI images from 3 planes `Sagittal` , `Coronal` , `Axial`.

In [ ]:
print(study_series_df.Anatomical_Plane.unique())

The `annotation_df` is from train csv which has labeled parameters which are knee abnormalities findings, they are labeled `1.0` for availbility of the finding and `0.0` for missing of the finding. we have `Report` which is a text report written by the doctors in different languages.

In [ ]:
display(annotation_df.head(3))

We merge ``annotations_df`` with ``study_series_df`` on ``StudyID``, which allows to have the labels and the anatomical planes in on single dataframe.

In [ ]:

"""
cols = [
    'StudyInstanceUID', 'SeriesInstanceUID','Anatomical_Plane', 'Report', 'ACL', 'MCL', 'Medial Meniscus',
    'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture',
    'Fluid_Sensitive', 'Fat_Suppression' , 'is_labeled'
]

merged_df = annotation_df.merge(right=study_series_df , on="StudyInstanceUID" ,how="inner")

merged_df =merged_df[cols]

"""



In [ ]:
# merged_df.to_csv("../data/knee_data.csv" , index=False)

In [ ]:
knee_df = pd.read_csv("../data/knee_data.csv")

The ``Knee_df`` contains all the necessary data provided , which allows to build model to predict abnormaility findings , but as you can see some of the series contain ``Null`` values. In the following section we try to interpolate Null values using the Reports provided.

In [ ]:
display(knee_df.head(3))

In [ ]:

series_path = f"{TRAIN_SEREIS_DICOM_PATH}/{knee_df.iloc[0].StudyInstanceUID}/{knee_df.iloc[0].SeriesInstanceUID}"

image, volume = tools.get_mri_volume(series_path)

print("Volume shape:", volume.shape)
print("Voxel spacing:", image.GetSpacing())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 3))

indices = [
  5 , 10 , 20
]

for ax, idx in zip(axes, indices):
    ax.imshow(volume[idx], cmap="gray")
    ax.set_title(f"Slice {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## Image & Reports Preprocessing

#### Extracting abnormality findings from Reports
Since some of the annotation provided in the `train.csv` are Null , we need to interpolate these data points by the inforamtion provided in the ``Reports`` column, The best method assumed for this process, is by using `LLM` model since the reports are in different languages and different wordings.

We use the ``LLM benchmarks API`` provided by the ``Kaggle`` to exratact annotation dictionaries from each report.

The kaggle API allows to have access to following models on local machine , there are also stronger models, which are only accesable on Kaggle website.

In [ ]:
load_dotenv()

models = os.getenv("LLMS_AVAILABLE", "")
print(models.split(",") if models else [])

In [ ]:
example_studies, evaluation_studies = llm_extractor.get_example_studies(annotation_df)
extractor = llm_extractor.KneeReportExtractor(
    example_text=llm_extractor.build_examples(example_studies),
    max_workers=3,
)

### Anotatting the reports
The following methods send a prompt which contain annotation task description and rules, definition of the variables, few shots examples ;  and recieves a dictionary of float values for each parameter.

In [ ]:
study = evaluation_studies.iloc[0]
prediction = extractor.predict(study["Report"])
print(prediction)

In [ ]:
results_df, errors = extractor.extract(knee_df)

display(results_df)
print(f"Extracted {len(results_df)} studies; {len(errors)} failed.")

In [ ]:
# results_df.to_csv("../data/reports_extract.csv")

### Compare Labels extracted by LLM & Labels provided by project Owner
We need to compare the studies that have been labeled by humans and the reports labeled by LLM to see the accuracy.

In [ ]:
reports_extract = pd.read_csv("../data/reports_extract_OpenAI.csv")

In [ ]:
label_cols = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

In [ ]:
reports_extract.StudyInstanceUID.isin(annotation_df.StudyInstanceUID).all()

In [ ]:
reports_extract = (
    reports_extract
    .set_index("StudyInstanceUID")
    .loc[annotation_df["StudyInstanceUID"]]
    .reset_index()
)

reports_extract = reports_extract.drop(columns=["Unnamed: 0"])

In [ ]:
display(reports_extract.head(3))

In [ ]:
labeled_df = annotation_df[
    annotation_df["is_labeled"] == True
].copy()


assert labeled_df["StudyInstanceUID"].is_unique
assert reports_extract["StudyInstanceUID"].is_unique


comparison_df = labeled_df[
    ["StudyInstanceUID"] + label_cols
].merge(
    reports_extract[["StudyInstanceUID"] + label_cols],
    on="StudyInstanceUID",
    how="inner",
    suffixes=("_human", "_llm"),
    validate="one_to_one"
)

print(f"Human-labeled studies: {len(labeled_df)}")
print(f"Matched studies: {len(comparison_df)}")

Using the `Accuracy`, `Precision`, `Recall`, `F1`, `ROC-AUC`  metrics for evaluating the varibles annotated by human and annotated by LLM

In [ ]:
threshold = 0.5

metrics = []

for col in label_cols:
    human = pd.to_numeric(
        comparison_df[f"{col}_human"], errors="coerce"
    )
    llm = pd.to_numeric(
        comparison_df[f"{col}_llm"], errors="coerce"
    )

    valid = human.notna() & llm.notna()

    y_true = human[valid]
    y_score = llm[valid]

    y_pred = (y_score >= threshold).astype(int)

    metrics.append({
        "Label": col,
        "Samples": len(y_true),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "ROC-AUC": (
            roc_auc_score(y_true, y_score)
            if y_true.nunique() == 2 else np.nan
        )
    })

metrics_df = pd.DataFrame(metrics)

display(metrics_df.round(3))

### Replacing Null values with LLM predicted labels

In [ ]:
knee_df = knee_df.set_index("StudyInstanceUID")
reports_extract = reports_extract.set_index("StudyInstanceUID")

knee_df = knee_df.fillna(reports_extract)

knee_df = knee_df.reset_index()

Setting values above 0.5 threshold to 1 and below 0.5 and equal to 0.5 to 0 

In [ ]:
set_binary = True

if set_binary:
    knee_df[knee_df[label_cols] > 0.5] = 1
    knee_df[knee_df[label_cols] <= 0.5] = 0
    

In [ ]:
# knee_df.to_csv("../data/knee_data_binary.csv", index=False)

## Build 3D CNN Fusion Model

Using annotated parameters provided by humans and LLM and MRI images, we can build a mulitple branched ``3D CNN fusion model`` which trains separately on different ``Anatomical planes`` and predict knee abnormalities.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
label_cols = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

In [6]:
knee_df = pd.read_csv("../data/knee_data_binary.csv")

In [7]:
BATCH_SIZE = 1
TEST_SIZE = 0.2
INIT_LR = 0.001
NUM_EPOCHS = 10

For preventing data leakage we use ``Groupshufflesplit`` on ``StudyUID``.

In [8]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(knee_df, groups=knee_df["StudyInstanceUID"])
)

train_df = knee_df.iloc[train_idx]
val_df = knee_df.iloc[val_idx]

In [9]:

class KneeStudyDataset(Dataset):
    def __init__(self, df, dicom_root, label_columns, transform=None):
        self.dicom_root = Path(dicom_root)
        self.label_columns = list(label_columns)
        self.transform = transform

        required = [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "Anatomical_Plane",
        ]
        
        if df[required].isna().any().any():
            raise ValueError("Study, series, and plane values must be present.")

        self.studies = []

        for study_uid, group in df.groupby("StudyInstanceUID", sort=False):
            
            if (group[self.label_columns].nunique(dropna=False) > 1).any():
                raise ValueError(f"Inconsistent labels in study {study_uid}")

            series = group.drop_duplicates("SeriesInstanceUID").copy()
            self.studies.append((str(study_uid), series))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, index):
        study_uid, series = self.studies[index]

        values = series.iloc[0][self.label_columns].to_numpy(
            dtype=np.float32
        )
        
        label_mask = np.isfinite(values)
        labels = np.where(label_mask, values, 0.0)

        planes = {}

        for _, row in series.iterrows():
            series_uid = str(row["SeriesInstanceUID"])
            plane = str(row["Anatomical_Plane"]).strip().lower()

            series_path = self.dicom_root / study_uid / series_uid

            image, volume = tools.get_mri_volume(str(series_path))

            # SimpleITK array: [D, H, W] -> tensor: [C, D, H, W]
            volume = torch.from_numpy(
                np.ascontiguousarray(volume, dtype=np.float32)
            ).unsqueeze(0)

            if self.transform is not None:
                volume = self.transform(volume)

            planes.setdefault(plane, []).append({
                "volume": volume,
                "series_uid": series_uid,
                "fluid_sensitive": row.get("Fluid_Sensitive"),
                "fat_suppression": row.get("Fat_Suppression"),
                # SimpleITK spacing is in (x, y, z) order.
                "spacing_xyz": image.GetSpacing(),
            })

        return {
            "study_uid": study_uid,
            "planes": planes,
            "labels": torch.from_numpy(labels),
            "label_mask": torch.from_numpy(label_mask),
        }


def collate_studies(batch):
    
    return {
        "study_uids": [sample["study_uid"] for sample in batch],
        "planes": [sample["planes"] for sample in batch],
        "labels": torch.stack([sample["labels"] for sample in batch]),
        "label_mask": torch.stack(
            [sample["label_mask"] for sample in batch]
        ),
    }
    

Since MRI volumes provided have different size and number of slices, we can resize the volumes using the transform and interpolate after resizing, but resizing can reduce resolution or produce unreliable voxels.

In [10]:
class Volume_transformer:
    def __init__(self, size=(32, 400 , 400)):
        self.size = size

    def __call__(self, volume):
        
        flat = volume.flatten()
        max_samples = 1_000_000
        step = max(1, (flat.numel() + max_samples - 1) // max_samples)
        sample = flat[::step]

        low, high = torch.quantile(
            sample,
            sample.new_tensor([0.01, 0.99])
        )

        volume = volume.clamp(low, high)
        volume = (
            (volume - volume.mean())
            / volume.std(unbiased=False).clamp_min(1e-6)
        )


        volume = F.interpolate(
            volume.unsqueeze(0),  
            size=self.size,
            mode="trilinear",
            align_corners=False,
        )
        return volume


In [ ]:
transform = Volume_transformer(size=(32, 256 , 256))

transform = None

train_set = KneeStudyDataset(
    train_df,
    TRAIN_SEREIS_DICOM_PATH,
    label_cols,
    transform=transform
)

val_set = KneeStudyDataset(
    val_df,
    TRAIN_SEREIS_DICOM_PATH,
    label_cols,
    transform=transform
)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_studies,
    num_workers=0,  
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_studies,
    num_workers=0,
)



In [12]:
sample = train_set[0]

for plane, series_list in sample["planes"].items():
    print(plane, len(series_list))
    for series in series_list:
        print(" ", series["series_uid"], series["volume"].shape)

sagittal 2
  1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282 torch.Size([1, 36, 400, 400])
  1.2.826.0.1.3680043.8.498.37372672584387115708078590109474719429 torch.Size([1, 36, 320, 320])
coronal 2
  1.2.826.0.1.3680043.8.498.12451507871565834868863189854063726441 torch.Size([1, 30, 384, 384])
  1.2.826.0.1.3680043.8.498.21497837050858589852101556928971904198 torch.Size([1, 30, 320, 320])
axial 1
  1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430 torch.Size([1, 32, 320, 320])


In [ ]:
sample

In [ ]:
batch = next(iter(train_loader))
batch

In [ ]:
class CNN3DEncoder(nn.Module):

    def __init__(self , inchannels = (1 , 32 , 64 , 128 , 256 ) , feature_dim = 128):
        super().__init__()

        pools = [
            (1, 2, 2),
            (2, 2, 2),
            (1, 2, 2)
            
        ]

        layers = []

        for in_ch, out_ch, pool in zip(
            inchannels[:-1], inchannels[1:], pools
        ):
            layers.extend([
                nn.Conv3d(
                    in_ch, out_ch,
                    kernel_size=3,
                    padding=1,
                    bias=False,
                ),
                nn.GroupNorm(8, out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool3d(pool),
            ])

        self.block = nn.Sequential(*layers)

        self.features = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(inchannels[-1], feature_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.features(self.block(x))



In [ ]:
class CNN3DFusion(nn.Module):

    def __init__(self ,inchannels = (1 , 32 , 64 , 128 , 256 ) , num_labels = 12 , feature_dim = 128):
        super().__init__()

        self.sagittal_encoder = CNN3DEncoder(inchannels=inchannels, feature_dim=feature_dim)
        self.coronal_encoder = CNN3DEncoder(inchannels=inchannels, feature_dim=feature_dim)
        self.Axial_encoder = CNN3DEncoder(inchannels=inchannels, feature_dim=feature_dim)

        
        self.head = nn.Sequential(
            nn.Linear(feature_dim * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels),
            
        
        )

    def forward(self , studies):

        device = next(self.parameters()).device

        encoders = {
            "sagittal": self.sagittal_encoder,
            "coronal": self.coronal_encoder,
            "axial": self.Axial_encoder,
        }

        study_outputs = []

        for study in studies:
            plane_features = []

            for plane in ("sagittal", "coronal", "axial"):
                volume_features = []

                for series in study[plane]:
                    volume = series["volume"].to(device)
                    features = encoders[plane](volume) 
                    volume_features.append(features)

                
                features = torch.stack(volume_features).mean(dim=0)
                plane_features.append(features)

            combined_features = torch.cat(plane_features, dim=1)
            study_outputs.append(self.head(combined_features))

        return torch.cat(study_outputs, dim=0) 
        


In [ ]:
VIEWS = (
    "sagittal",
    "coronal",
    "axial"
)

model = CNN3DFusion(inchannels= (1, 64 , 128 , 256) , num_labels = len(label_cols) , feature_dim = 128).to(device)

optimizer = optim.Adam(model.parameters(), lr=INIT_LR)
criterion = nn.BCEWithLogitsLoss()

history = {"train_loss" : [] , "val_loss": []}


for e in range(NUM_EPOCHS):
    model.train()

    train_loss_sum = 0.0
    train_count = 0

    for batch in tqdm(train_loader, desc=f"Train Epoch {e + 1}"):
        labels = batch["labels"].float().to(device)

        optimizer.zero_grad()

        predict = model(batch["planes"])
        train_loss = criterion(predict, labels)

        train_loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        train_loss_sum += train_loss.item() * batch_size
        train_count += batch_size

    model.eval()

    val_loss_sum = 0.0
    val_count = 0

    with torch.inference_mode():
        for batch in tqdm(val_loader, desc=f"Validate Epoch {e + 1}"):
            labels = batch["labels"].float().to(device)

            predict = model(batch["planes"])
            val_loss = criterion(predict, labels)

            batch_size = labels.size(0)
            val_loss_sum += val_loss.item() * batch_size
            val_count += batch_size

    train_bce = train_loss_sum / train_count
    val_bce = val_loss_sum / val_count

    history["train_loss"].append(train_bce)
    history["val_loss"].append(val_bce)

    print(
        f"Epoch {e + 1}/{NUM_EPOCHS} | "
        f"Train BCE: {train_bce:.4f} | "
        f"Val BCE: {val_bce:.4f}"
    )
    


In [ ]:
model.eval()

all_labels = []
all_probabilities = []

loss_sum = 0.0
label_count = 0

with torch.inference_mode():
    for batch in val_loader:
        labels = batch["labels"].float().to(device)
        logits = model(batch["planes"])
        probabilities = torch.sigmoid(logits)

        assert probabilities.shape == labels.shape

        losses = F.binary_cross_entropy(
            probabilities, labels, reduction="none"
        )

        loss_sum += losses.sum().item()
        label_count += labels.numel()

        all_labels.append(labels.cpu().numpy())
        all_probabilities.append(probabilities.cpu().numpy())

y_true = np.concatenate(all_labels)
y_prob = np.concatenate(all_probabilities)
y_pred = (y_prob >= 0.5).astype(int)

print(f"Validation BCE: {loss_sum / label_count:.4f}")